In [ ]:
# Built-in modules
import re
from pathlib import Path

# Core scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests

# Setup autoreload
%load_ext autoreload
%autoreload 2

# Local module imports
import microscopy_analysis.d00_utils.utilities as utils
import microscopy_analysis.d04_plot_data.restructure_data as rd
import microscopy_analysis.d04_plot_data.plot_timelapse_data as ptd
import microscopy_analysis.d04_plot_data.run_stats as rs

# Jupyter settings
%matplotlib inline

In [ ]:
# set default style for graphs
smallpts_fillcolor = '#DBDBDB'
smallpts_edgecolor = '#AFAFAF'
ticks_fontsize = 8
axislabel_fontsize = 10
linewidth = 0.5

rc = {'svg.fonttype':'none', 
      'font.family':'Arial',
      'figure.dpi': 150,
      'axes.linewidth': linewidth,
      'axes.labelweight':'bold',
      'axes.labelsize': axislabel_fontsize,
      'axes.labelpad':7.5
     }

sns.set(rc)
sns.set_style("ticks")

# Color palettes
lat_color = '#66c2a5'
jasp_color = '#fc8d62'
ctrl_color = '#8da0cb'
drugtx_palette = [ctrl_color, lat_color, jasp_color]

ctrl_color = '#dd8452'
lat_color = '#4c72b0'
ctrllat_palette = [ctrl_color, lat_color]

caax_color = '#A218A2'
cmp_color = '#116F11'
cmpreg_palette = [caax_color, cmp_color]

xmin = -1
xmax = 11

timelapse_figsize=(4,3)
scatter_figsize=(2.5, 3)

In [ ]:
#df_path = Path(input('Please enter the full path for the dataframe:\n'))
df_path = Path('/Users/kwu2/Library/CloudStorage/GoogleDrive-kwu2@stanford.edu/My Drive/Lab/OL_compaction/Experiments_compiled/actin_pharmacological/data/combined_analysis.csv')

In [ ]:
df = pd.read_csv(df_path)
df.head()

In [ ]:
# Filter out missing data/inconsistent timepoints
df.loc[df['elapsed time (hr)']== -0.5, 'elapsed time (hr)'] = -1
df_filt = rd.filter_incomplete_data(df, 'elapsed time (hr)', max_num_incomplete=2)

# Compute changes in compaction values between timepoints
base_cmp_cols = ['cell area', 'CAAX-positive area', 'compacted area', '% compaction']
df_filt, cmp_cols = rd.compute_change_cols(df_filt, base_cmp_cols)
                                        
# Average compaction-related values by biorep
groupbycols = ['tx', 'experiment', 'elapsed time (hr)']
biorep_cmp_df = rd.compute_means_by_biorep(df_filt, groupbycols, cmp_cols, omit_col='omit')
biorep_cmp_df_path = df_path.parent / 'biorep_cmp_data.csv'
utils.safe_save_csv(biorep_cmp_df, biorep_cmp_df_path)

# Make graphs

In [ ]:
graphs_dirpath = Path('/Users/kwu2/Library/CloudStorage/GoogleDrive-kwu2@stanford.edu/My Drive/Lab/OL_compaction/Experiments_compiled/actin_pharmacological/graphs')

In [ ]:
stats_combined = pd.DataFrame()
stats_df_path = df_path.parent / 'stats_df.csv'

## Control actin plots

In [ ]:
# --- Normalize actin intensity columns ---

actin_df = df_filt.copy()

# get initial actin intensity of each cell (will normalize by this)
norm_ref = df_filt.groupby('UID')['mean actin int (cell)'].transform('first')

# normalize all of the actin columns by the reference values
actin_cols = [
    col for col in df_filt.columns 
    if 'actin int' in col and
    (not col.startswith('norm '))
]

actin_df_norm = actin_df[actin_cols].div(norm_ref, axis=0) * 100
actin_df_norm = actin_df_norm.add_prefix('norm ')
norm_actin_cols = actin_df_norm.columns.tolist()
ycols = cmp_cols + actin_cols + norm_actin_cols
actin_df = pd.concat([actin_df, actin_df_norm], axis=1)

# --- Compute biological replicate means for actin intensity ---
actingroupbycols = groupbycols + ['time relative to compaction (hr)']
biorep_actin_df = rd.compute_means_by_biorep(actin_df, actingroupbycols, ycols, omit_col='actin omit')
biorep_actin_df_path = df_path.parent / 'biorep_actin_data.csv'
utils.safe_save_csv(biorep_actin_df, biorep_actin_df_path)

In [ ]:
time_col = 'elapsed time (hr)'

ctrl_biorep_actin_df = biorep_actin_df[biorep_actin_df['tx']=='DMSO']
ctrl_biorep_actin_df = rd.select_timepoints(ctrl_biorep_actin_df, time_col, 0, 10)

prefix = 'ctrl_actinint_'
data = ctrl_biorep_actin_df
ycol = 'mean actin int (a.u.) / cell'
subject = 'experiment'
group_category = 'region'

norm_actin_cols = [col for col in data.columns if 'norm mean' in col]

paired_actin_cols = [('norm mean actin int (caax)', 'norm mean actin int (compacted)'), ('norm mean actin int in uncompacted regions that stay uncompacted next frame', 'norm mean actin int in uncompacted regions that will compact next frame')]
paired_actin_labels = [('non-compact', 'compact'), ('stays non-compact', 'compacts next frame')]


for ycols, labels in zip(paired_actin_cols, paired_actin_labels):
    comparison_label = {
        'columns': ycols,
        'labels': labels,
        'name': f'{prefix}_{ptd.clean_column_name(labels[0])}_vs_{ptd.clean_column_name(labels[1])}'
    }

    # Re-arrange dataframe and run stats
    df_long, stats_df = rs.run_repeated_measures_stats(
        df=data,
        subject=subject,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=True
    )
    # add stats to combined stats dataframe
    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    labels = [ptd.wrap_text(label) for label in labels]

    # plot timelapse graphs
    ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=ycols,
        palette=cmpreg_palette,
        group_labels=labels,
        graphname=comparison_label['name'],
        save_dir=graphs_dirpath,
        stats_df=stats_df, 
        figsize=timelapse_figsize
    )
    
    # Plot scatter plots for first and last timepoints
    timepoints = sorted(df_long[time_col].dropna().unique())
    tp_list = [timepoints[0], timepoints[-1]]
        
    for tp in tp_list:
        ptd.plot_individual_tp(
            df_long=df_long,
            stats_df=stats_df,
            tp=tp,
            xcol=time_col,
            ycol=ycol,
            group_col=group_category,
            comparison_label=comparison_label['name'],
            labels=labels,
            palette=cmpreg_palette,
            savepath=graphs_dirpath / f"{comparison_label['name']}_tp{tp:.0f}.svg"
        )
        
    utils.safe_save_csv(stats_combined, stats_df_path)

In [ ]:
# --- Normalize actin intensity columns ---

#actin_before_cmp_df = df_filt.copy()
print(len(actin_before_cmp_df['UID'].unique()))
actin_before_cmp_df = df[(df['tx']=='DMSO') & (df['actin omit']!='Y')].copy()
print(len(actin_before_cmp_df['UID'].unique()))

# # Filter out incomplete data
time_col = 'time relative to compaction (hr)'
value_col = 'mean actin int leading up to cmp'
actin_before_cmp_df = rd.filter_incomplete_data(actin_before_cmp_df, time_col=time_col, value_col=value_col, max_num_incomplete=2)
actin_before_cmp_df

# get initial actin intensity of each cell (will normalize by this)
norm_ref = actin_before_cmp_df.groupby('UID')['mean actin int (cell)'].transform('first')

# normalize all of the actin columns by the reference values
actinbeforecmp_cols = [
    col for col in actin_before_cmp_df.columns 
    if ('leading up to' in col or 'remain CAAX+' in col) and
    (not col.startswith('norm '))
]

actin_before_cmp_df_norm = actin_before_cmp_df[actinbeforecmp_cols].div(norm_ref, axis=0) * 100
actin_before_cmp_df_norm = actin_before_cmp_df_norm.add_prefix('norm ')
norm_actinbeforecmp_cols = actin_before_cmp_df_norm.columns.tolist()
ycols = base_cmp_cols + actinbeforecmp_cols + norm_actinbeforecmp_cols
actin_before_cmp_df = pd.concat([actin_before_cmp_df, actin_before_cmp_df_norm], axis=1)

# --- Compute biological replicate means for actin intensity ---
actingroupbycols = groupbycols + ['time relative to compaction (hr)']
biorep_actin_before_cmp_df = rd.compute_means_by_biorep(actin_before_cmp_df, actingroupbycols, ycols, omit_col='actin omit')
biorep_actin_before_cmp_df_path = df_path.parent / 'biorep_actin_before_cmp_data.csv'
utils.safe_save_csv(biorep_actin_before_cmp_df, biorep_actin_before_cmp_df_path)

In [ ]:
prefix = 'ctrl_actinbeforecmp_'
data = biorep_actin_before_cmp_df
ycol = 'mean actin int (a.u.) / cell'
subject = 'experiment'
group_category = 'region'

paired_actin_cols = [('norm mean actin int in regions that remain CAAX+', 'norm mean actin int leading up to cmp')]
paired_actin_labels = [('stays non-compact', 'compacts')]


for ycols, labels in zip(paired_actin_cols, paired_actin_labels):
    comparison_label = {
        'columns': ycols,
        'labels': labels,
        'name': f'{prefix}_{ptd.clean_column_name(labels[0])}_vs_{ptd.clean_column_name(labels[1])}'
    }

    # Re-arrange dataframe and run stats
    df_long, stats_df = rs.run_repeated_measures_stats(
        df=data,
        subject=subject,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=True
    )
    # add stats to combined stats dataframe
    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    labels = [ptd.wrap_text(label) for label in labels]

    # plot timelapse graphs
    ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=ycols,
        palette=cmpreg_palette,
        group_labels=labels,
        graphname=comparison_label['name'],
        save_dir=graphs_dirpath,
        stats_df=stats_df, 
        figsize=timelapse_figsize
    )
    
    # Plot scatter plots for first and last timepoints
    timepoints = sorted(df_long[time_col].dropna().unique())
    tp_list = [timepoints[0], timepoints[-1]]
        
    for tp in tp_list:
        ptd.plot_individual_tp(
            df_long=df_long,
            stats_df=stats_df,
            tp=tp,
            xcol=time_col,
            ycol=ycol,
            group_col=group_category,
            comparison_label=comparison_label['name'],
            labels=labels,
            palette=cmpreg_palette,
            savepath=graphs_dirpath / f"{comparison_label['name']}_tp{tp:.0f}.svg"
        )
        
    utils.safe_save_csv(stats_combined, stats_df_path)


## Control compaction plots

In [ ]:
# ctrl_biorep_cmp_df = biorep_actin_df[biorep_actin_df['tx']=='DMSO']
# ctrl_biorep_cmp_df = rd.select_timepoints(ctrl_biorep_cmp_df, time_col, 0, 10)

ctrl_biorep_cmp_df = biorep_cmp_df[biorep_cmp_df['tx']=='DMSO']
ctrl_biorep_cmp_df = rd.select_timepoints(ctrl_biorep_cmp_df, time_col, 0, 10)

prefix = 'ctrl_cmp'
data = ctrl_biorep_cmp_df
time_col = 'elapsed time (hr)'
subject = 'experiment'
group_category = 'tx'
hue_order = ['DMSO']
palette = [ctrl_color]

# # plot timelapse graphs
# for ycol in cmp_cols:
    
#     ptd.plot_timelapse_lines(
#         df_long=data,
#         xcol=time_col,
#         ycol=ycol,
#         hue=group_category,
#         hue_order=hue_order,
#         palette=palette,
#         graphname=f'{prefix}_{ptd.clean_column_name(ycol)}',
#         save_dir=graphs_dirpath,
#         stats_df=stats_df, 
#         figsize=timelapse_figsize
#     )


data = data.sort_values([subject, time_col])
g = data.groupby([subject])

first = g.first()
last = g.last()

dt = last[time_col] - first[time_col]

# Calculate rate of change for all y columns
rates = (last[cmp_cols] - first[cmp_cols]).div(dt, axis=0)

# mean and sd across replicates
summary = pd.DataFrame({
    "mean_rate": rates.mean(),
    "sd_rate": rates.std()
})

print(summary)

In [ ]:
data = df_filt[df_filt['omit']!='Y']
xcol = 'change in compacted area'
ycol = 'change in cell area'

sns.scatterplot(data=data, x=xcol, y=ycol)

# seaborn.lineplot(data=None, *, x=None, y=None, hue=None, size=None, style=None, units=None, weights=None, palette=None, hue_order=None, hue_norm=None, sizes=None, size_order=None, size_norm=None, dashes=True, markers=None, style_order=None, estimator='mean', errorbar=('ci', 95), n_boot=1000, seed=None, orient='x', sort=True, err_style='band', err_kws=None, legend='auto', ci='deprecated', ax=None, **kwargs)

# Control vs latrunculin plots

In [ ]:
# # --- Compute biological replicate means for actin int leading up to compaction ---

# Create a copy of the dataframe with values for actin leading up to compaction
actin_before_cmp_df = df[df['tx']=='DMSO'].copy()
time_col = 'time relative to compaction (hr)'
value_col = 'mean actin int leading up to cmp'
actin_before_cmp_df = rd.filter_incomplete_data(actin_before_cmp_df, time_col, value_col=value_col, max_num_incomplete=2)


# Average compaction-related values by biorep
groupbycols = ['tx', 'experiment', time_col]
actinbeforecmp_cols = [value_col, 'mean actin int in regions that remain CAAX+']
ctrl_biorep_actinbeforecmp_df = rd.compute_means_by_biorep(actin_before_cmp_df, groupbycols, actinbeforecmp_cols, omit_col='actin omit')
ctrl_biorep_actinbeforecmp_df_path = df_path.parent / 'ctrl_biorep_actinbeforecmp_data.csv'
utils.safe_save_csv(ctrl_biorep_actinbeforecmp_df, ctrl_biorep_actinbeforecmp_df_path)

ctrl_biorep_actinbeforecmp_df.head()

In [ ]:
ctrlvslat_biorep_cmp_df = biorep_cmp_df[biorep_cmp_df['tx'].isin(['DMSO', 'latA'])]

prefix = 'ctrlvslat_cmp'
data = ctrlvslat_biorep_cmp_df
time_col = 'elapsed time (hr)'
subject = 'experiment'
group_category = 'tx'
hue_order = ['DMSO', 'latA']
palette = ctrllat_palette

for ycol in cmp_cols:
    comparison_label = {
        'columns': ycol,
        'labels': hue_order,
        'name': f'{prefix}_{ptd.clean_column_name(ycol)}'
    }
    
    # Run stats
    df_long, stats_df = rs.run_repeated_measures_stats(
        df=data,
        subject=subject,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=False
    )

    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    # plot timelapse graphs
    ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=hue_order,
        palette=palette,
        tx_line=0,
        graphname=comparison_label['name'],
        save_dir=graphs_dirpath,
        stats_df=stats_df, 
        figsize=timelapse_figsize
    )
        
    utils.safe_save_csv(stats_combined, stats_df_path)


In [ ]:
ctrlvslat_biorep_cmp_df = biorep_cmp_df[biorep_cmp_df['tx'].isin(['DMSO', 'latA', 'jasp'])]

prefix = 'ctrlvslat_cmp'
data = ctrlvslat_biorep_cmp_df
time_col = 'elapsed time (hr)'
subject = 'experiment'
group_category = 'tx'
hue_order = ['DMSO', 'latA', 'jasp']
palette = drugtx_palette

for ycol in cmp_cols:
    comparison_label = {
        'columns': ycol,
        'labels': hue_order,
        'name': f'{prefix}_{ptd.clean_column_name(ycol)}'
    }
    
    # Run stats
    df_long, stats_df = rs.run_repeated_measures_stats(
        df=data,
        subject=subject,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=False
    )

    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    # plot timelapse graphs
    ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=hue_order,
        palette=palette,
        tx_line=0,
        graphname=comparison_label['name'],
        save_dir=graphs_dirpath,
        stats_df=stats_df, 
        figsize=timelapse_figsize
    )
        
    utils.safe_save_csv(stats_combined, stats_df_path)


In [ ]:
cmp_zone_df_path = Path('/Users/kwu2/Library/CloudStorage/GoogleDrive-kwu2@stanford.edu/My Drive/Lab/OL_compaction/Experiments_compiled/actin_pharmacological/data/cmp_zone_analysis_combined.csv')

In [ ]:
cmp_zone_df = pd.read_csv(cmp_zone_df_path)
cmp_zone_df.head()

In [ ]:
# filter compaction zone df
time_col = 'elapsed time (hr)'

cmp_zone_df.loc[cmp_zone_df[time_col]== -0.5, time_col] = -1
cmp_zone_df_filt = rd.filter_incomplete_data(cmp_zone_df, time_col, max_num_incomplete=2)
cmp_zone_df_filt = rd.select_timepoints(cmp_zone_df_filt, time_col, -1, 10)
cmp_zone_df_filt.head()

In [ ]:
#------Calculate mean number and area of compaction zones by cell------
groupbycols = ['tx', 'experiment', time_col]
sum_cmp_groupby = groupbycols + ['UID']

cmp_area_col = 'area (microns^2)'
sum_cmp_zone_df = cmp_zone_df_filt.groupby(sum_cmp_groupby, as_index=False).agg({
    cmp_area_col: ['mean', 'count']
})

# Flatten MultiIndex columns
sum_cmp_zone_df.columns = ['_'.join(col).strip() if col[1] else col[0] for col in sum_cmp_zone_df.columns.values]

# Rename columns
cmp_zone_ycols = ['mean cmp zone area (μm²)', 'num cmp zones']
sum_cmp_zone_df = sum_cmp_zone_df.rename(columns={
    f'{cmp_area_col}_mean': cmp_zone_ycols[0],
    f'{cmp_area_col}_count': cmp_zone_ycols[1]
})

# Compute changes in compaction values between timepoints
sum_cmp_zone_df, cmp_zone_ycols = rd.compute_change_cols(sum_cmp_zone_df, cmp_zone_ycols, time_col=time_col)

sum_cmp_zone_df.head()

In [ ]:
#------Calculate averages by biological replicates------
biorep_sum_cmp_zone_df = rd.compute_means_by_biorep(sum_cmp_zone_df, groupbycols, cmp_zone_ycols)
biorep_sum_cmp_zone_df.to_csv(df_path.parent/'biorep_sum_cmp_zone_df.csv')
biorep_sum_cmp_zone_df.head()

In [ ]:
prefix = 'ctrlvslatA'
data = biorep_sum_cmp_zone_df[biorep_sum_cmp_zone_df['tx'].isin(['DMSO', 'latA'])]
xcol = 'elapsed time (hr)'

time_col = 'elapsed time (hr)'
cmp_area_col = 'area (microns^2)'

cum_change_cmp_cols = [col for col in cmp_zone_ycols if 'cumulative change' in col]
hue_order = ['DMSO', 'latA']
subject = 'experiment'
time = time_col
group_category = 'tx'
value = ycol


for ycol in cum_change_cmp_cols:

    comparison_label = {
        'columns': ycols,
        'labels': hue_order,
        'name': f'{prefix}_{ptd.clean_column_name(ycol)}'
    }

    # Calculate stats
    df_long, stats_df = rs.run_repeated_measures_stats(data, subject, time, group_category, ycol, comparison_label, melt_df=False)
    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    # Plot timelapse data
    ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=hue_order,
        palette=ctrllat_palette,
        group_labels=None,
        graphname=comparison_label['name'],
        save_dir=graphs_dirpath,
        stats_df=stats_df, 
    )
    print(stats_df)


In [ ]:
stats_combined